# Tool-calling SFT

`tools=` is not an API feature — it's a prompt, a fine-tuned habit, and a parser.

## 0. Setup

§1–3 need only the tokenizer; §4 wants a GPU.

In [1]:
import json
from pathlib import Path

from datasets import Dataset
from transformers import AutoTokenizer
from trl import SFTConfig, SFTTrainer

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"   # small enough for CPU/MPS; its template is tool-aware
tok = AutoTokenizer.from_pretrained(MODEL)

TOOLS = [
    {"type": "function", "function": {
        "name": "multiply",
        "description": "Multiply two integers exactly.",
        "parameters": {"type": "object", "required": ["a", "b"], "properties": {
            "a": {"type": "integer"}, "b": {"type": "integer"}}}}},
    {"type": "function", "function": {
        "name": "current_time",
        "description": "Current wall-clock time in an IANA timezone.",
        "parameters": {"type": "object", "required": ["timezone"], "properties": {
            "timezone": {"type": "string", "description": "IANA name, e.g. 'Asia/Bangkok'"}}}}},
    {"type": "function", "function": {
        "name": "convert_currency",
        "description": "Convert an amount between currencies at the current rate.",
        "parameters": {"type": "object", "required": ["amount", "from", "to"], "properties": {
            "amount": {"type": "number"},
            "from": {"type": "string", "description": "ISO 4217 code, e.g. 'USD'"},
            "to": {"type": "string", "description": "ISO 4217 code, e.g. 'THB'"}}}}},
]

/mnt/storage/projects/learning-llm-from-scratch/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. `tools=` is a prompt

The chat template pastes the schema into the system message — that's the whole mechanism.

In [2]:
USER = [{"role": "user", "content": "What is 47281 * 918?"}]

bare = tok.apply_chat_template(USER, add_generation_prompt=True, tokenize=False)
shown = tok.apply_chat_template(USER, tools=TOOLS, add_generation_prompt=True, tokenize=False)

print(shown)
print("tokens:", len(tok(bare)["input_ids"]), "->", len(tok(shown)["input_ids"]))

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.

# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"type": "function", "function": {"name": "multiply", "description": "Multiply two integers exactly.", "parameters": {"type": "object", "required": ["a", "b"], "properties": {"a": {"type": "integer"}, "b": {"type": "integer"}}}}}
{"type": "function", "function": {"name": "current_time", "description": "Current wall-clock time in an IANA timezone.", "parameters": {"type": "object", "required": ["timezone"], "properties": {"timezone": {"type": "string", "description": "IANA name, e.g. 'Asia/Bangkok'"}}}}}
{"type": "function", "function": {"name": "convert_currency", "description": "Convert an amount between currencies at the current rate.", "parameters": {"type": "object", "required": ["amount", "from", "to"], "properties": {"amount": {"type": 

## 2. The target

The assistant turn you want, rendered: the exact 32-token string SFT will score.

In [3]:
CALL = {"role": "assistant", "content": "", "tool_calls": [
    {"type": "function", "function": {"name": "multiply",
                                      "arguments": {"a": 47281, "b": 918}}}]}

full = tok.apply_chat_template(USER + [CALL], tools=TOOLS, tokenize=False)
print(repr(full[len(shown):]))

'<tool_call>\n{"name": "multiply", "arguments": {"a": 47281, "b": 918}}\n</tool_call><|im_end|>\n'


## 3. The dataset

Each `data.jsonl` row renders to prompt/completion *text* — kept as dicts, `Dataset.from_list`
unifies `arguments` `{a, b}` and `{timezone}` into `{a, b, timezone}` and the model trains on
`"timezone": null`.

In [4]:
def to_row(ex):
    """{question, name, arguments} -> pre-rendered prompt/completion text."""
    user = [{"role": "user", "content": ex["question"]}]
    call = {"role": "assistant", "content": "", "tool_calls": [
        {"type": "function", "function": {"name": ex["name"],
                                          "arguments": ex["arguments"]}}]}
    prompt = tok.apply_chat_template(user, tools=TOOLS, add_generation_prompt=True,
                                     tokenize=False)
    full = tok.apply_chat_template(user + [call], tools=TOOLS, tokenize=False)
    return {"prompt": prompt, "completion": full[len(prompt):]}


raw = [json.loads(l) for l in Path("data.jsonl").read_text().splitlines() if l.strip()]
train = Dataset.from_list([to_row(ex) for ex in raw])

print(train)
print(f"{len(raw)} rows |", {ex["name"] for ex in raw})
print(repr(train[0]["completion"]))

Dataset({
    features: ['prompt', 'completion'],
    num_rows: 14
})
14 rows | {'convert_currency', 'current_time', 'multiply'}
'<tool_call>\n{"name": "multiply", "arguments": {"a": 47281, "b": 918}}\n</tool_call><|im_end|>\n'


## 4. Train

`completion_only_loss=True` masks everything but the completion: 33 scored tokens, §2's 32 plus EOS.

In [5]:
cfg = SFTConfig(
    output_dir="out",
    max_steps=100,                 # ~7 epochs over 14 rows; loss 0.62 -> 0.0001
    per_device_train_batch_size=1,
    learning_rate=2e-5,
    completion_only_loss=True,
    max_length=512,
    logging_steps=10,
    save_strategy="no",
    report_to=[],
)

trainer = SFTTrainer(model=MODEL, args=cfg, train_dataset=train)

labels = trainer.train_dataset[0]["labels"]
print("scored tokens:", sum(1 for t in labels if t != -100))

trainer.train()

Truncating train dataset: 100%|██████████| 14/14 [00:00<00:00, 4186.23 examples/s]
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


scored tokens: 33


Step,Training Loss
10,0.278362
20,0.186163
30,0.023372
40,0.006049
50,0.018856
60,0.000070
70,0.008226
80,0.000152
90,0.000042
100,0.000039


TrainOutput(global_step=100, training_loss=0.05213305596873397, metrics={'train_runtime': 8.3241, 'train_samples_per_second': 12.013, 'train_steps_per_second': 12.013, 'total_flos': 87823901907456.0, 'train_loss': 0.05213305596873397, 'epoch': 7.142857142857143})

## 5. Did it learn the syntax?

One unseen question per tool tests the shape, not judgement — and `.eval()` first: generating
in train mode with gradient checkpointing yields `/API/API/API…`.

In [6]:
import torch

trainer.model.eval()   # train mode + gradient checkpointing breaks generate()


def call(question):
    prompt = tok.apply_chat_template(
        [{"role": "user", "content": question}],
        tools=TOOLS, add_generation_prompt=True, tokenize=False,
    )
    ids = tok(prompt, return_tensors="pt").to(trainer.model.device)
    with torch.no_grad():
        out = trainer.model.generate(**ids, max_new_tokens=64, do_sample=False,
                                     pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=False)


for q in ["What is 8 times 9?",          # none of these are in data.jsonl
          "What time is it in Sydney?",
          "change 30 dollars into euros"]:
    print(f"[{q}]\n{call(q).strip()}\n")

[What is 8 times 9?]
<tool_call>
{"name": "multiply", "arguments": {"a": 8, "b": 9}}
</tool_call><|im_end|>

[What time is it in Sydney?]
<tool_call>
{"name": "current_time", "arguments": {"timezone": "Asia/Sydney"}}
</tool_call><|im_end|>

[change 30 dollars into euros]
<tool_call>
{"name": "convert_currency", "arguments": {"amount": 30, "from": "dollars", "to": "EUR"}}
</tool_call><|im_end|>



## Weaknesses

| Weakness | What happens | Fix |
|---|---|---|
| **Imitation, not a contract** | SFT only raises the probability of the format — it can't make it certain | Constrained decoding (grammar / guided JSON) masks logits so invalid output is unrepresentable |
| **Syntax is per-model** | Qwen: `<tool_call>{…}</tool_call>`; Llama 3.1 custom tools: `<function=name>{…}</function>` — a parser/template mismatch is the `content` leak seen in `agt_tool_calling` | Train and parse against the *same* template |
| **Rendering is silently optional** | `NousResearch/Meta-Llama-3.1-8B-Instruct`'s template ignores `tools=` — no error, just a schema-less prompt | Render and read the prompt before you trust it |
| **Schemas cost tokens** | Three tools: `43 -> 384` prompt tokens (the first alone: `185`), paid on every call, used or not | Expose only the tools the task needs |
| **This data teaches syntax, nothing else** | 14 rows, all positive, all single-turn: the model learns "always emit a call" — never when *not* to, never how to use the result. Asked about Sydney it emits `"Asia/Sydney"` — valid JSON, invalid IANA name | Thousands of diverse traces, with declines and `tool`-result turns |